# Aing_리그전 Transformer (Multi30k)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aing-gachon/26-Spring-Transformer-Study/blob/main/Week3/Aing_%E1%84%85%E1%85%B5%E1%84%80%E1%85%B3%E1%84%8C%E1%85%A5%E1%86%AB_Transformer_Finetune.ipynb)

> 목표: **Transformer 구조와 학습 하이퍼파라미터** 를 바꾸어 번역 성능을 비교하고, Encoder–Decoder·Attention·FFN의 역할을 코드로 체험합니다.  
> 최종 점수(리그전): **Multi30k test BLEU** *(best valid checkpoint 기준)*

---

## 리그전 규칙

### FIXED (공정성/평가 계약 — 변경 금지)
- 환경: **Colab NVIDIA T4**, seed **42**
- 데이터: Multi30k 독일어→영어, **Train 16,000 / Valid 1,014**, 동일 tokenizer
- 모델 크기: **학습 파라미터 800만 이하**, 사전학습 가중치 사용 안 함
- 학습 예산: **최대 4,000 updates**, 유효 batch **64** (16 × accumulation 4)
- 학습 절차: **warmup 100 → cosine**, label smoothing 0.1, gradient clipping 1.0
- Early stopping: **500 updates마다 valid BLEU 평가**, 최소 **1,500 updates 이후 3회 연속 0.1점 초과 개선이 없으면 종료**
- Final/Test: 학습 중 **가장 높은 valid BLEU**의 모델을 불러와 test 평가
- 제출: `full`의 최대 예산 도달 또는 공통 early stopping으로 정상 종료한 실험

### TUNE (STEP5의 ARCH / TRAIN_HP에서 한 번에 설정)
- 모델 폭, Encoder/Decoder 층 수, Attention head 수, FFN 비율, Pre-LN/Post-LN
- **Optimizer (Adam / AdamW), learning rate, weight decay, dropout**
- `quick`/`full`과 재개 여부는 실행 방식이며 공통 규칙은 바뀌지 않습니다.

---

**실행 안내**  
처음에는 위에서부터 실행하세요. 이후 **STEP5 → STEP7**을 실행하면 다른 구조를 비교할 수 있습니다. 함수 정의 셀은 접혀 있으며 필요할 때 펼쳐 읽을 수 있습니다.

**속도 참고**  
기존 1,200 updates 실측은 회당 약 3분이었습니다. 이번 최대 4,000 updates 설정의 시간과 상위 구조 간 점수 차이는 T4 추가 측정이 필요합니다. Early stopping에 따라 실제 학습량은 달라집니다.



### STEP1. 환경 확인
- **설명:** Colab에 설치된 PyTorch/CUDA를 사용하고, 토크나이저와 평가기 버전을 통일합니다.
- `tokenizers`는 문장을 토큰으로 나누고, `sacrebleu`는 번역 결과를 채점합니다.
- **실행:** 설치 셀과 import 셀을 차례로 실행합니다. 이미 다른 버전을 import했다면 커널을 다시 시작하세요.


In [ ]:
%pip -q install "tokenizers==0.22.2" "sacrebleu==2.6.0"

In [ ]:
import os, json, math, time, random, hashlib, platform, gc, copy, zipfile
from pathlib import Path
from contextlib import nullcontext
from urllib.request import urlopen
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors
import tokenizers, sacrebleu


### STEP2. seed / device / 공통 규칙
- **설명:** 모든 참가자가 같은 데이터와 실행 조건에서 비교하도록 FIXED 값을 정의합니다.
- `SEED_FIXED=42`는 모델 초기화와 난수의 기준입니다. 데이터 추출 seed도 공통입니다.
- `PROTOCOL`에는 최대 updates, batch, 파라미터 상한, 평가·early stopping 규칙이 들어갑니다.
- **확인:** 출력 GPU가 **NVIDIA T4**인지 확인하세요. learning rate 등 튜닝 값은 STEP5에서 설정합니다.


In [ ]:
PROTOCOL = dict(
    version="architecture-league-v3-tuning",
    dataset_revision="4589883f3d09d4ef6361784e03f0ead219836469",
    train_samples=16000,
    data_seed=42,
    vocab_size=8000,
    max_length=96,
    micro_batch=16,
    accumulation=4,
    total_steps=4000,
    quick_steps=500,
    eval_every=500,
    warmup_steps=100,
    label_smoothing=0.1,
    grad_clip=1.0,
    max_parameters=8000000,
    max_memory_gib=12.0,
    seed=42,
    early_stop_min_steps=1500,
    early_stop_patience=3,
    early_stop_min_delta=0.1,
    generation_max_tokens=96,
    eval_batch=32,
    tokenizer_version=tokenizers.__version__,
    sacrebleu_version=sacrebleu.__version__,
)
PAD, BOS, EOS, UNK = 0, 1, 2, 3
SPECIALS = ["<pad>", "<bos>", "<eos>", "<unk>"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT = Path("./transformer_league_runs_v3")
SEED_FIXED = 42


def digest(value):
    return hashlib.sha256(
        json.dumps(value, sort_keys=True, ensure_ascii=False).encode()
    ).hexdigest()


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def environment():
    return dict(
        python=platform.python_version(),
        torch=torch.__version__,
        cuda=torch.version.cuda,
        gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        tokenizers=tokenizers.__version__,
        sacrebleu=sacrebleu.__version__,
    )


def require_t4():
    if not torch.cuda.is_available() or "T4" not in torch.cuda.get_device_name(0):
        raise RuntimeError(
            "공식 실험은 Colab NVIDIA T4에서 실행하세요. 커널/런타임 GPU를 확인하세요."
        )
    print(json.dumps(environment(), indent=2))


In [ ]:
require_t4()


### STEP3. Multi30k 로드 + 고정 Split/Subsample
- **설명:** 독일어 문장을 영어로 번역하는 Multi30k의 train 16,000쌍과 validation 전체를 준비합니다.
- 동일 revision과 seed로 train을 추출하며, 이미 다운로드한 데이터는 캐시를 사용합니다.
- Tokenizer는 train 문장으로만 학습하고, 학습 토큰은 최대 96개로 제한합니다. BLEU 정답은 잘리지 않은 원문을 사용합니다.
- **확인:** 출력의 데이터 수와 `DATA_PACKAGE`를 확인하세요. 구조나 TRAIN_HP를 바꿔도 같은 데이터 패키지를 사용합니다.


In [ ]:
def prepare_data(cache_dir="./league_data"):
    cache = Path(cache_dir)
    cache.mkdir(parents=True, exist_ok=True)
    raw, source_hashes = {}, {}
    for split, filename in [("train", "train.jsonl"), ("validation", "val.jsonl")]:
        path = cache / (PROTOCOL["dataset_revision"] + "_" + filename)
        if not path.exists():
            url = f"https://huggingface.co/datasets/bentrevett/multi30k/resolve/{PROTOCOL['dataset_revision']}/{filename}"
            with urlopen(url, timeout=90) as response:
                payload = response.read()
            temp = path.with_suffix(".tmp")
            temp.write_bytes(payload)
            temp.replace(path)
        payload = path.read_bytes()
        source_hashes[split] = hashlib.sha256(payload).hexdigest()
        raw[split] = [json.loads(line) for line in payload.decode().splitlines() if line.strip()]
        for i, row in enumerate(raw[split]):
            if not all(isinstance(row.get(k), str) and row[k].strip() for k in ("de", "en")):
                raise ValueError(f"{split}[{i}]에 빈 문장이나 잘못된 데이터가 있습니다.")
    ids = (
        np.random.default_rng(PROTOCOL["data_seed"])
        .permutation(len(raw["train"]))[: PROTOCOL["train_samples"]]
        .tolist()
    )
    selected = [raw["train"][i] for i in ids]
    tok = Tokenizer(models.BPE(unk_token="<unk>"))
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=True)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=PROTOCOL["vocab_size"],
        min_frequency=2,
        special_tokens=SPECIALS,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    )
    tok.train_from_iterator((row[k] for row in selected for k in ("de", "en")), trainer=trainer)
    assert [tok.token_to_id(t) for t in SPECIALS] == [PAD, BOS, EOS, UNK]
    for text in ["the cat is playing outside.", "Über die Straße!  two spaces", "I'm learning."]:
        assert tok.decode(tok.encode(text).ids) == text, "토크나이저 round-trip 실패"
    tok_hash = hashlib.sha256(tok.to_str().encode()).hexdigest()
    tok.save(str(cache / f"tokenizer_{tok_hash}.json"))

    def encode(rows):
        result = []
        clipped_src = clipped_tgt = 0
        for row in rows:
            src = tok.encode(row["de"]).ids
            tgt = tok.encode(row["en"]).ids
            clipped_src += len(src) + 2 > PROTOCOL["max_length"]
            clipped_tgt += len(tgt) + 2 > PROTOCOL["max_length"]
            # EOS survives truncation. Full references stay separate and are never shortened.
            result.append(
                dict(
                    src=[BOS] + src[: PROTOCOL["max_length"] - 2] + [EOS],
                    tgt=[BOS] + tgt[: PROTOCOL["max_length"] - 2] + [EOS],
                    reference=row["en"],
                )
            )
        return result, dict(count=len(rows), source_clipped=clipped_src, target_clipped=clipped_tgt)

    train, tr_stats = encode(selected)
    valid, va_stats = encode(raw["validation"])
    manifest = dict(
        protocol=copy.deepcopy(PROTOCOL),
        source_hashes=source_hashes,
        train_ids_hash=digest(ids),
        tokenizer_hash=tok_hash,
        actual_vocab=tok.get_vocab_size(),
        train=tr_stats,
        validation=va_stats,
    )
    package_id = digest(manifest)
    print("DATA_PACKAGE", package_id, json.dumps({"train": tr_stats, "validation": va_stats}))
    return dict(
        train=train, validation=valid, tokenizer=tok, manifest=manifest, package_id=package_id
    )


In [ ]:
DATA = prepare_data()


### STEP4. 모델 구현 (Encoder–Decoder 구조)

포인트:
- `LeagueTransformer`: 처음부터 학습하는 `nn.Transformer` 기반 번역 모델
- `encode` / `decode`: source 표현 생성 / 이전 target을 보고 다음 토큰 예측
- 공유 Embedding과 Sinusoidal 위치 정보, Multi-Head Attention, FFN을 사용합니다.
- Causal mask는 미래 target을 차단하고, PAD mask는 padding을 제외합니다.
- STEP5의 `ARCH`로 폭·층 수·head·FFN·정규화 위치를 정하고, `TRAIN_HP['dropout']`을 Embedding과 Transformer 내부에 적용합니다.
- `build_optimizer`는 선택한 Adam/AdamW와 learning rate·weight decay를 적용합니다.
- **실행:** 이 셀은 함수 정의입니다. 실제 모델 생성과 파라미터·mask·메모리 검사는 STEP7에서 수행합니다.


In [ ]:
def validate_training_hp(hp=None):
    """STEP5 설정을 검증하고 실험에 사용할 독립적인 사본을 반환합니다."""
    hp = TRAIN_HP if hp is None else hp
    if not isinstance(hp, dict) or set(hp) != {
        "optimizer",
        "learning_rate",
        "weight_decay",
        "dropout",
    }:
        raise ValueError(
            "TRAIN_HP에는 optimizer, learning_rate, weight_decay, dropout을 지정하세요."
        )
    if not isinstance(hp["optimizer"], str) or hp["optimizer"] not in ("Adam", "AdamW"):
        raise ValueError("optimizer는 Adam 또는 AdamW입니다.")
    result = dict(optimizer=hp["optimizer"])
    for key in ("learning_rate", "weight_decay", "dropout"):
        value = hp[key]
        if type(value) not in (int, float) or not math.isfinite(value):
            raise ValueError(f"{key}는 유한한 숫자여야 합니다.")
        result[key] = float(value)
    if result["learning_rate"] <= 0 or result["weight_decay"] < 0 or not 0 <= result["dropout"] < 1:
        raise ValueError("learning_rate > 0, weight_decay >= 0, 0 <= dropout < 1이어야 합니다.")
    return result


def build_optimizer(model, hp):
    hp = validate_training_hp(hp)
    optimizer_class = {"Adam": torch.optim.Adam, "AdamW": torch.optim.AdamW}[hp["optimizer"]]
    return optimizer_class(
        model.parameters(),
        lr=hp["learning_rate"],
        betas=(0.9, 0.98),
        eps=1e-9,
        weight_decay=hp["weight_decay"],
    )


ALLOWED = dict(
    d_model={128, 192, 256, 384},
    encoder_layers={1, 2, 3, 4},
    decoder_layers={1, 2, 3, 4},
    num_heads={2, 4, 8},
    ffn_ratio={2, 4},
    norm_first={False, True},
)


def validate_arch(arch):
    if set(arch) != set(ALLOWED):
        raise ValueError(f"ARCH 키는 {list(ALLOWED)}만 허용합니다.")
    for key, choices in ALLOWED.items():
        expected = bool if key == "norm_first" else int
        if type(arch[key]) is not expected or arch[key] not in choices:
            raise ValueError(f"{key}: 허용값 {choices}")
    if arch["d_model"] % arch["num_heads"]:
        raise ValueError("d_model은 num_heads로 나누어떨어져야 합니다.")


class LeagueTransformer(nn.Module):
    def __init__(self, arch, vocab, training_hp=None):
        super().__init__()
        validate_arch(arch)
        self.arch = copy.deepcopy(arch)
        self.training_hp = validate_training_hp(training_hp)
        d = arch["d_model"]
        self.embedding = nn.Embedding(vocab, d, padding_idx=PAD)
        position = torch.arange(PROTOCOL["max_length"] + 2).float().unsqueeze(1)
        rate = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe = torch.zeros(PROTOCOL["max_length"] + 2, d)
        pe[:, 0::2] = torch.sin(position * rate)
        pe[:, 1::2] = torch.cos(position * rate)
        self.register_buffer("position", pe)
        self.dropout = nn.Dropout(self.training_hp["dropout"])
        self.transformer = nn.Transformer(
            d_model=d,
            nhead=arch["num_heads"],
            num_encoder_layers=arch["encoder_layers"],
            num_decoder_layers=arch["decoder_layers"],
            dim_feedforward=d * arch["ffn_ratio"],
            dropout=self.training_hp["dropout"],
            activation="relu",
            batch_first=True,
            norm_first=arch["norm_first"],
        )
        self.output = nn.Linear(d, vocab, bias=False)
        self.output.weight = self.embedding.weight
        # Shared embeddings have a controlled scale; default N(0,1) is too large for tied logits.
        nn.init.normal_(self.embedding.weight, mean=0.0, std=d**-0.5)
        with torch.no_grad():
            self.embedding.weight[PAD].zero_()
        count = sum(p.numel() for p in self.parameters())
        if count > PROTOCOL["max_parameters"]:
            raise ValueError(f"{count:,} parameters: 상한 초과")

    def embed(self, ids):
        if ids.shape[1] > len(self.position):
            raise ValueError("위치 인코딩 최대 길이 초과")
        return self.dropout(
            self.embedding(ids) * math.sqrt(self.arch["d_model"]) + self.position[: ids.shape[1]]
        )

    def encode(self, src):
        return self.transformer.encoder(self.embed(src), src_key_padding_mask=src.eq(PAD))

    def decode(self, trg, memory, src_pad):
        # nn.Transformer의 bool mask는 True가 차단입니다(Week2 허용 마스크와 반대).
        causal = torch.ones(trg.shape[1], trg.shape[1], device=trg.device, dtype=torch.bool).triu(1)
        hidden = self.transformer.decoder(
            self.embed(trg),
            memory,
            tgt_mask=causal,
            tgt_key_padding_mask=trg.eq(PAD),
            memory_key_padding_mask=src_pad,
        )
        return self.output(hidden)

    def forward(self, src, trg):
        return self.decode(trg, self.encode(src), src.eq(PAD))


def pad_batch(rows):
    src = nn.utils.rnn.pad_sequence(
        [torch.tensor(r["src"]) for r in rows], batch_first=True, padding_value=PAD
    )
    tgt = nn.utils.rnn.pad_sequence(
        [torch.tensor(r["tgt"]) for r in rows], batch_first=True, padding_value=PAD
    )
    return src.to(DEVICE), tgt.to(DEVICE)


def amp_context():
    return torch.autocast("cuda", dtype=torch.float16) if DEVICE.type == "cuda" else nullcontext()


def smoke_check(arch, data, official=True, training_hp=None):
    if official:
        require_t4()
    seed_all(42)
    hp = validate_training_hp(training_hp)
    model = LeagueTransformer(arch, data["tokenizer"].get_vocab_size(), hp).to(DEVICE)
    model.eval()
    src = torch.tensor([[BOS, 4, 5, EOS], [BOS, 6, 7, EOS]], device=DEVICE)
    tgt = torch.tensor([[BOS, 8, 9], [BOS, 10, 11]], device=DEVICE)
    with torch.no_grad():
        logits = model(src, tgt)
        changed = tgt.clone()
        changed[:, 2] = 12
        assert logits.shape == (2, 3, data["tokenizer"].get_vocab_size())
        assert torch.allclose(
            logits[:, :2], model(src, changed)[:, :2], atol=1e-4, rtol=1e-4
        ), "미래 target 누출"
        padded = F.pad(src, (0, 2), value=PAD)
        assert torch.allclose(logits, model(padded, tgt), atol=1e-4, rtol=1e-4), "source PAD 누출"
    model.train()
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    # Maximum training lengths, actual micro batch, full vocab projection and optimizer state.
    x = torch.randint(
        4,
        data["tokenizer"].get_vocab_size(),
        (PROTOCOL["micro_batch"], PROTOCOL["max_length"]),
        device=DEVICE,
    )
    x[:, 0] = BOS
    x[:, -1] = EOS
    optimizer = build_optimizer(model, hp)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda", init_scale=1024.0)
    start = time.perf_counter()
    with amp_context():
        loss = F.cross_entropy(
            model(x, x[:, :-1]).reshape(-1, data["tokenizer"].get_vocab_size()),
            x[:, 1:].reshape(-1),
        )
    assert torch.isfinite(loss), "loss NaN/Inf"
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    norm = nn.utils.clip_grad_norm_(model.parameters(), PROTOCOL["grad_clip"])
    assert torch.isfinite(norm), "gradient NaN/Inf"
    scaler.step(optimizer)
    scaler.update()
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    memory = torch.cuda.max_memory_reserved() / 2**30 if DEVICE.type == "cuda" else None
    if memory is not None and memory > PROTOCOL["max_memory_gib"]:
        raise RuntimeError(f"T4 메모리 예산 초과: {memory:.2f} GiB")
    result = dict(
        parameters=sum(p.numel() for p in model.parameters()),
        max_memory_gib=memory,
        max_length_micro_step_seconds=time.perf_counter() - start,
        loss=float(loss.detach()),
    )
    del model, optimizer, scaler
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    print("SMOKE PASS", result)
    return result


### STEP5. ARCH / TRAIN_HP (구조·학습 설정, TUNE)

**이 설정 셀 한 곳에서 모든 튜닝 값을 변경하세요.**

- **모델 구조 `ARCH`**
  - `d_model`: 토큰 표현의 폭 (128 / 192 / 256 / 384)
  - `encoder_layers` / `decoder_layers`: 각 stack의 층 수 (1~4)
  - `num_heads`: Attention head 수 (2 / 4 / 8)
  - `ffn_ratio`: FFN 중간 차원 / d_model (2 / 4)
  - `norm_first`: False는 Post-LN, True는 Pre-LN
- **학습 설정 `TRAIN_HP`**
  - `optimizer`: **AdamW / Adam** 중 선택합니다.
  - `learning_rate`: warmup 이후 도달하는 최대 학습률입니다. 이후 공통 cosine 스케줄을 따릅니다.
  - `weight_decay`: 가중치 정규화 강도입니다. Adam과 AdamW의 적용 방식이 다르므로 같은 값이 같은 효과를 뜻하지 않습니다.
  - `dropout`: 학습 중 일부 특징을 끄는 확률입니다. Embedding과 Transformer 내부에 적용되며 평가에서는 꺼집니다.

**실행 방식:** `quick`은 첫 500 updates, `full`은 최대 4,000 updates 또는 early stopping까지 실행합니다. `RESUME=True`일 때 구조와 학습 설정이 모두 같으면 이어서 실행합니다. 하나라도 바꾸면 새 실험으로 저장됩니다.

기준값은 출발점이며 최적값을 보장하지 않습니다. 먼저 한 항목씩 바꾸어 영향을 확인하세요. 허용 구조라도 **800만 파라미터를 넘으면 사용할 수 없습니다.**


In [ ]:
# === TUNE 1: Transformer 구조 ===
ARCH = dict(
    d_model=256,  # 128 / 192 / 256 / 384
    encoder_layers=3,  # 1 / 2 / 3 / 4
    decoder_layers=3,  # 1 / 2 / 3 / 4
    num_heads=4,  # 2 / 4 / 8
    ffn_ratio=4,  # 2 / 4
    norm_first=False,  # False: Post-LN / True: Pre-LN
)

# === TUNE 2: 학습 하이퍼파라미터 ===
TRAIN_HP = dict(
    optimizer="AdamW",  # "AdamW" / "Adam"
    learning_rate=1e-3,  # warmup 이후 도달할 최대 learning rate (> 0)
    weight_decay=0.01,  # 가중치 정규화 강도 (>= 0)
    dropout=0.1,  # Embedding·Attention·FFN 등의 dropout 확률 (0 <= p < 1)
)

# === 실행 방식 ===
LEAGUE_MODE = "quick"  # quick: 500 updates / full: 최대 4,000 updates
RESUME = True  # 구조와 TRAIN_HP가 모두 같을 때 이어서 실행


### STEP6. 학습 설정 + train/eval
- **설명:** STEP5에서 선택한 구조와 학습 설정으로 teacher forcing 학습과 greedy 번역 평가를 수행합니다. 이 단계의 함수 코드를 수정할 필요는 없습니다.
- **TUNE 반영:** Adam/AdamW, 최대 learning rate, weight decay, dropout은 `TRAIN_HP`에서 가져옵니다. Adam 계열의 betas=(0.9,0.98), eps=1e-9는 공통입니다.
- **FIXED:** seed·데이터·유효 batch·최대 updates·scheduler 형태·label smoothing·평가·early stopping 규칙을 유지합니다.
- 500 updates마다 validation BLEU를 평가하며, 최고 점수 모델은 `best.pt`에 저장합니다.
- 1,500 updates 이후 기준 점수보다 **0.1점 초과 개선**이 없는 평가가 3회 연속이면 종료합니다. 작은 개선은 기준 대비 누적될 수 있으며, 가장 이른 조기 종료는 2,500 updates입니다.
- 최고 모델 저장은 patience와 별개입니다. 0.1점보다 작은 개선도 최고 점수라면 저장합니다.
- `latest.pt`는 모델·optimizer·난수·patience와 **TRAIN_HP**를 보존합니다. 다른 튜닝 설정의 checkpoint를 이어 학습하지 않습니다.

아래 평가·학습·결과 관리 함수 셀은 처음 한 번 실행하면 됩니다.


In [ ]:
@torch.inference_mode()
def translate(model, rows):
    model.eval()
    src, _ = pad_batch(rows)
    with amp_context():
        memory = model.encode(src)
        out = torch.full((len(rows), 1), BOS, device=DEVICE, dtype=torch.long)
        done = torch.zeros(len(rows), device=DEVICE, dtype=torch.bool)
        for _ in range(PROTOCOL["generation_max_tokens"]):
            logits = model.decode(out, memory, src.eq(PAD))[:, -1].float()
            logits[:, [PAD, BOS, UNK]] = -torch.inf
            nxt = logits.argmax(-1)
            nxt = torch.where(done, torch.full_like(nxt, EOS), nxt)
            out = torch.cat([out, nxt[:, None]], 1)
            done |= nxt.eq(EOS)
            if done.all():
                break
    sequences = []
    for ids in out[:, 1:].cpu().tolist():
        sequences.append(ids[: ids.index(EOS)] if EOS in ids else ids)
    return sequences


@torch.inference_mode()
def evaluate_model(model, rows, tok):
    if not rows:
        raise ValueError("평가 데이터가 비어 있습니다.")
    start = time.perf_counter()
    predictions = []
    for i in range(0, len(rows), PROTOCOL["eval_batch"]):
        predictions += tok.decode_batch(translate(model, rows[i : i + PROTOCOL["eval_batch"]]))
    refs = [r["reference"] for r in rows]
    metric = sacrebleu.metrics.BLEU(tokenize="13a", lowercase=False, effective_order=False)
    score = metric.corpus_score(predictions, [refs])
    chrf = sacrebleu.corpus_chrf(predictions, [refs]).score
    return dict(
        bleu=score.score,
        chrf=chrf,
        signature=str(metric.get_signature()),
        count=len(refs),
        seconds=time.perf_counter() - start,
        examples=[dict(prediction=p, reference=r) for p, r in zip(predictions[:3], refs[:3])],
    )


def prepare_test_data(tok, cache_dir="./league_data"):
    cache = Path(cache_dir)
    cache.mkdir(parents=True, exist_ok=True)
    path = cache / (PROTOCOL["dataset_revision"] + "_test.jsonl")
    if not path.exists():
        url = f"https://huggingface.co/datasets/bentrevett/multi30k/resolve/{PROTOCOL['dataset_revision']}/test.jsonl"
        with urlopen(url, timeout=90) as response:
            payload = response.read()
        temp = path.with_suffix(".tmp")
        temp.write_bytes(payload)
        temp.replace(path)
    payload = path.read_bytes()
    raw = [json.loads(line) for line in payload.decode().splitlines() if line.strip()]
    if not raw or any(
        not isinstance(r.get(k), str) or not r[k].strip() for r in raw for k in ("de", "en")
    ):
        raise ValueError("잘못된 test 데이터입니다.")
    rows = []
    for row in raw:
        rows.append(
            dict(
                src=[BOS] + tok.encode(row["de"]).ids[: PROTOCOL["max_length"] - 2] + [EOS],
                tgt=[BOS] + tok.encode(row["en"]).ids[: PROTOCOL["max_length"] - 2] + [EOS],
                reference=row["en"],
                source=row["de"],
            )
        )
    return rows, hashlib.sha256(payload).hexdigest()


@torch.inference_mode()
def evaluate_loss(model, rows):
    # Teacher-forced, unsmoothed cross entropy per non-PAD target token.
    # BLEU generation remains separate and receives no target tokens.
    model.eval()
    loss_sum = 0.0
    token_count = 0
    for i in range(0, len(rows), PROTOCOL["eval_batch"]):
        src, tgt = pad_batch(rows[i : i + PROTOCOL["eval_batch"]])
        with amp_context():
            logits = model(src, tgt[:, :-1])
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                tgt[:, 1:].reshape(-1),
                ignore_index=PAD,
                reduction="sum",
            )
        loss_sum += float(loss)
        token_count += int(tgt[:, 1:].ne(PAD).sum())
    if not token_count:
        raise ValueError("평가할 target 토큰이 없습니다.")
    return loss_sum / token_count


def evaluate_best_checkpoint(result, data, cache_dir="./league_data"):
    identity = {
        k: result[k] for k in ("architecture", "training_hp", "seed", "package_id", "protocol")
    }
    if result["protocol"] != PROTOCOL or result["package_id"] != data["package_id"]:
        raise ValueError("평가 데이터/프로토콜 불일치")
    if digest(identity)[:16] != result["run_id"]:
        raise ValueError("잘못된 run_id")
    folder = ROOT / result["run_id"]
    state = torch.load(folder / "best.pt", map_location="cpu", weights_only=True)
    best_record = max(result["history"], key=lambda row: row["bleu"])
    if (
        state["identity"] != identity
        or state["step"] != best_record["step"]
        or state["metrics"]["bleu"] != best_record["bleu"]
    ):
        raise ValueError("validation 최고 체크포인트 불일치")
    model = LeagueTransformer(
        result["architecture"], data["tokenizer"].get_vocab_size(), result["training_hp"]
    ).to(DEVICE)
    model.load_state_dict(state["model"], strict=True)
    test_rows, test_hash = prepare_test_data(data["tokenizer"], cache_dir)
    valid = evaluate_model(model, data["validation"], data["tokenizer"])
    valid["loss"] = evaluate_loss(model, data["validation"])
    test = evaluate_model(model, test_rows, data["tokenizer"])
    test["loss"] = evaluate_loss(model, test_rows)
    for row, example in zip(test_rows, test["examples"]):
        example["source"] = row["source"]
    final = dict(
        run_id=result["run_id"],
        training_hp=copy.deepcopy(result["training_hp"]),
        selected_step=state["step"],
        seed=result["seed"],
        completed_steps=result["completed_steps"],
        full_budget_completed=result["completed_steps"] == PROTOCOL["total_steps"],
        training_completed=training_completed(result),
        stop_reason=result["stop_reason"],
        environment=environment(),
        test_sha256=test_hash,
        validation=valid,
        test=test,
    )
    (folder / "final_scores.json").write_text(json.dumps(final, ensure_ascii=False, indent=2))
    del model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return final


In [ ]:
# === Early stopping: validation만 사용 ===
def early_stopping_state(history):
    """0.1점 초과 개선을 기준으로 patience를 계산합니다. 최고 모델 저장은 별도입니다."""
    reference = -math.inf
    bad_evals = 0
    stop_step = None
    for record in history:
        score = record["bleu"]
        if not math.isfinite(score) or not 0 <= score <= 100:
            raise ValueError("validation BLEU가 유효하지 않습니다.")
        if score > reference + PROTOCOL["early_stop_min_delta"]:
            reference = score
            bad_evals = 0
        elif record["step"] >= PROTOCOL["early_stop_min_steps"]:
            bad_evals += 1
        if bad_evals >= PROTOCOL["early_stop_patience"]:
            stop_step = record["step"]
            break
    return dict(reference=reference, bad_evals=bad_evals, stop_step=stop_step)


def completion_reason(history, completed_steps):
    """실제 평가 이력으로 정상 종료 여부를 판정합니다. 미완료는 None입니다."""
    if type(completed_steps) is not int or not 0 <= completed_steps <= PROTOCOL["total_steps"]:
        raise ValueError("잘못된 학습 진행 기록입니다.")
    expected = list(range(PROTOCOL["eval_every"], completed_steps + 1, PROTOCOL["eval_every"]))
    if [r["step"] for r in history] != expected:
        raise ValueError("validation 평가 시점이 공통 규칙과 다릅니다.")
    state = early_stopping_state(history)
    if state["stop_step"] is not None and state["stop_step"] < completed_steps:
        raise ValueError("early stopping 이후 학습한 기록입니다.")
    if completed_steps == PROTOCOL["total_steps"]:
        return "max_steps"
    if state["stop_step"] == completed_steps:
        return "early_stopping"
    return None


def training_completed(result):
    """quick/수동 중단을 제외하고 최대 예산 또는 공통 조기 종료를 인정합니다."""
    if result["protocol"] != PROTOCOL or result["seed"] != SEED_FIXED:
        return False
    validate_training_hp(result["training_hp"])
    reason = completion_reason(result["history"], result["completed_steps"])
    return result["mode"] == "full" and reason is not None and result.get("stop_reason") == reason


def lr_factor(step):
    warm = PROTOCOL["warmup_steps"]
    total = PROTOCOL["total_steps"]
    if step < warm:
        return (step + 1) / max(1, warm)
    return 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))


def rng_state():
    return dict(
        python=random.getstate(),
        numpy=np.random.get_state(),
        torch=torch.get_rng_state(),
        cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    )


def restore_rng(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"].cpu())
    if state["cuda"] is not None:
        torch.cuda.set_rng_state_all([x.cpu() for x in state["cuda"]])


def atomic_save(value, path):
    path = Path(path)
    tmp = path.with_suffix(".tmp")
    torch.save(value, tmp)
    tmp.replace(path)


def train_run(arch, data, mode="quick", seed=42, resume=True, official=True, training_hp=None):
    if mode not in ("quick", "full"):
        raise ValueError("mode는 quick 또는 full입니다.")
    if type(seed) is not int or seed != SEED_FIXED:
        raise ValueError("리그전 seed는 42로 고정합니다.")
    if official:
        require_t4()
    validate_arch(arch)
    hp = validate_training_hp(training_hp)
    seed_all(seed)
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    identity = dict(
        architecture=copy.deepcopy(arch),
        training_hp=copy.deepcopy(hp),
        seed=seed,
        package_id=data["package_id"],
        protocol=copy.deepcopy(PROTOCOL),
    )
    run_id = digest(identity)[:16]
    folder = ROOT / run_id
    folder.mkdir(parents=True, exist_ok=True)
    latest = folder / "latest.pt"
    best_path = folder / "best.pt"
    model = LeagueTransformer(arch, data["tokenizer"].get_vocab_size(), hp).to(DEVICE)
    optimizer = build_optimizer(model, hp)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda", init_scale=1024.0)
    step = 0
    best = -1.0
    history = []
    train_seconds = 0.0
    eval_seconds = 0.0
    batch_size = PROTOCOL["micro_batch"] * PROTOCOL["accumulation"]
    if latest.exists():
        if not resume:
            raise FileExistsError("기존 실험이 있습니다. resume=True로 이어서 실행하세요.")
        # Only load this notebook's own trusted checkpoint; never load untrusted .pt files.
        state = torch.load(latest, map_location="cpu", weights_only=False)
        if state["identity"] != identity:
            raise ValueError("체크포인트의 구조/데이터/학습 설정이 다릅니다.")
        if state["environment"] != environment():
            raise ValueError("재개 환경이 달라졌습니다. 같은 라이브러리와 T4 환경을 사용하세요.")
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scaler.load_state_dict(state["scaler"])
        step = state["step"]
        best = state["best"]
        history = state["history"]
        train_seconds = state["train_seconds"]
        eval_seconds = state["eval_seconds"]
        restore_rng(state["rng"])
        if state["early_stopping"] != early_stopping_state(history):
            raise ValueError("early stopping 재개 상태가 평가 이력과 다릅니다.")
        print(f"재개: {run_id} / {step} updates", flush=True)
    target = PROTOCOL["quick_steps"] if mode == "quick" else PROTOCOL["total_steps"]
    if step > target:
        raise ValueError("이미 quick 구간을 넘긴 실험입니다. full 모드로 확인하세요.")
    stop_reason = completion_reason(history, step)
    permutations = {}
    rows = data["train"]
    size = len(rows)

    def batch_for(update):
        ids = []
        for offset in range(batch_size):
            epoch, index = divmod(update * batch_size + offset, size)
            if epoch not in permutations:
                permutations[epoch] = np.random.default_rng(
                    PROTOCOL["data_seed"] + epoch
                ).permutation(size)
            ids.append(int(permutations[epoch][index]))
        # Same effective batch for every architecture; sort only to reduce micro-batch padding.
        return sorted([rows[i] for i in ids], key=lambda r: max(len(r["src"]), len(r["tgt"])))

    def save_latest():
        atomic_save(
            dict(
                identity=identity,
                environment=environment(),
                model=model.state_dict(),
                optimizer=optimizer.state_dict(),
                scaler=scaler.state_dict(),
                step=step,
                best=best,
                history=history,
                train_seconds=train_seconds,
                eval_seconds=eval_seconds,
                rng=rng_state(),
                early_stopping=early_stopping_state(history),
            ),
            latest,
        )

    while step < target and stop_reason is None:
        batch = batch_for(step)
        denominator = sum(len(r["tgt"]) - 1 for r in batch)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        for group in optimizer.param_groups:
            group["lr"] = hp["learning_rate"] * lr_factor(step)
        start = time.perf_counter()
        loss_sum = 0.0
        for offset in range(0, batch_size, PROTOCOL["micro_batch"]):
            src, tgt = pad_batch(batch[offset : offset + PROTOCOL["micro_batch"]])
            with amp_context():
                logits = model(src, tgt[:, :-1])
                loss = (
                    F.cross_entropy(
                        logits.reshape(-1, logits.size(-1)),
                        tgt[:, 1:].reshape(-1),
                        ignore_index=PAD,
                        label_smoothing=PROTOCOL["label_smoothing"],
                        reduction="sum",
                    )
                    / denominator
                )
            if not torch.isfinite(loss):
                raise FloatingPointError("loss NaN/Inf: 실험 중단")
            scaler.scale(loss).backward()
            loss_sum += float(loss.detach())
        scaler.unscale_(optimizer)
        norm = nn.utils.clip_grad_norm_(model.parameters(), PROTOCOL["grad_clip"])
        if not torch.isfinite(norm):
            raise FloatingPointError("gradient NaN/Inf: 학습률/정밀도 점검 필요")
        scaler.step(optimizer)
        scaler.update()
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        train_seconds += time.perf_counter() - start
        step += 1
        if step == 1:
            print(f"학습 시작: {run_id} / 목표 {target:,} updates", flush=True)
        if step % PROTOCOL["eval_every"] == 0 or step == PROTOCOL["total_steps"]:
            metrics = evaluate_model(model, data["validation"], data["tokenizer"])
            eval_seconds += metrics["seconds"]
            history.append(dict(step=step, loss=loss_sum, **metrics))
            if metrics["bleu"] > best:
                best = metrics["bleu"]
                atomic_save(
                    dict(identity=identity, model=model.state_dict(), step=step, metrics=metrics),
                    best_path,
                )
            stop_reason = completion_reason(history, step)
            patience = early_stopping_state(history)["bad_evals"]
            print(
                f"{step:>4}/{PROTOCOL['total_steps']} | loss {loss_sum:.3f} | valid BLEU {metrics['bleu']:.2f} | best {best:.2f} | patience {patience}/{PROTOCOL['early_stop_patience']}",
                flush=True,
            )
        if step % 100 == 0 or step == target or stop_reason is not None:
            save_latest()
    summary = dict(
        run_id=run_id,
        **identity,
        mode=mode,
        completed_steps=step,
        best_valid_bleu=best,
        stop_reason=stop_reason or "quick_limit",
        early_stopping=early_stopping_state(history),
        parameters=sum(p.numel() for p in model.parameters()),
        train_seconds=train_seconds,
        eval_seconds=eval_seconds,
        peak_memory_gib=torch.cuda.max_memory_reserved() / 2**30 if DEVICE.type == "cuda" else None,
        environment=environment(),
        history=history,
        official_environment=official and DEVICE.type == "cuda",
    )
    (folder / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2))
    data["tokenizer"].save(str(folder / "tokenizer.json"))
    (folder / "data_manifest.json").write_text(
        json.dumps(data["manifest"], ensure_ascii=False, indent=2)
    )
    print(
        f"학습 종료: {summary['stop_reason']} / {step:,} updates / best valid BLEU {best:.2f}",
        flush=True,
    )
    del optimizer, scaler, model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return summary


In [ ]:
# === 실험 비교와 제출 ===
def compare_runs():
    records = []
    for path in ROOT.glob("*/summary.json"):
        result = json.loads(path.read_text())
        if result.get("protocol") == PROTOCOL and result.get("seed") == SEED_FIXED:
            records.append(result)
    records.sort(key=lambda r: r["best_valid_bleu"], reverse=True)
    print(
        f"{'run_id':16}  {'mode':5}  {'steps':>5}  {'BLEU':>6}  {'params(M)':>9}  optimizer / lr / decay / dropout  종료"
    )
    for r in records:
        print(
            f"{r['run_id']}  {r['mode']:5}  {r['completed_steps']:5}  {r['best_valid_bleu']:6.2f}  {r['parameters']/1e6:9.2f}  {r['training_hp']['optimizer']} / {r['training_hp']['learning_rate']:g} / {r['training_hp']['weight_decay']:g} / {r['training_hp']['dropout']:g}  {r['stop_reason']}"
        )
    return records


def export_submission(run_id):
    if (
        not isinstance(run_id, str)
        or len(run_id) != 16
        or any(c not in "0123456789abcdef" for c in run_id)
    ):
        raise ValueError("잘못된 run_id")
    folder = ROOT / run_id
    s = json.loads((folder / "summary.json").read_text())
    if not training_completed(s) or not s["official_environment"]:
        raise ValueError(
            "T4 full 정상 종료(최대 updates 또는 early stopping) 실험만 제출할 수 있습니다."
        )
    if s["protocol"] != PROTOCOL:
        raise ValueError("리그 프로토콜 불일치")
    names = ["best.pt", "summary.json", "tokenizer.json", "data_manifest.json"]
    hashes = {name: hashlib.sha256((folder / name).read_bytes()).hexdigest() for name in names}
    (folder / "checksums.json").write_text(json.dumps(hashes, indent=2))
    path = ROOT / f"submission_{run_id}.zip"
    with zipfile.ZipFile(path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for name in names + ["checksums.json"]:
            z.write(folder / name, arcname=name)
    print("SUBMISSION", path)
    return path


### STEP7. 학습 실행 + 최종 스코어보드 출력
- **설명:** 설정 검증과 최대 길이 backward 검사를 거쳐 STEP5의 조합으로 학습합니다.
- 최종 리그 점수는 **test BLEU**입니다. 모델 선택과 early stopping에는 validation만 사용합니다.
- 최고 validation checkpoint를 다시 불러와 valid/test loss·BLEU·chrF와 구조·학습 설정·종료 사유를 출력합니다.
- 정상 early stopping도 `full` 완료로 인정합니다. `quick`은 초기 확인용이며 제출할 수 없습니다.
- **반복 실험:** STEP5를 수정·실행한 뒤 이 STEP7을 다시 실행하세요. 데이터와 함수 정의를 다시 실행할 필요는 없습니다.
- 같은 설정의 완료된 실험은 추가 학습 없이 불러옵니다. 최종 조합을 결정한 뒤 아래 제출 셀의 주석을 해제하여 ZIP을 생성하세요.


In [ ]:
SMOKE = smoke_check(ARCH, DATA, training_hp=TRAIN_HP)
RESULT = train_run(
    ARCH, DATA, mode=LEAGUE_MODE, seed=SEED_FIXED, resume=RESUME, training_hp=TRAIN_HP
)
_ = compare_runs()

FINAL = evaluate_best_checkpoint(RESULT, DATA)
FINAL_VALID_LOSS = FINAL["validation"]["loss"]
FINAL_VALID_BLEU = FINAL["validation"]["bleu"]
FINAL_TEST_LOSS = FINAL["test"]["loss"]
FINAL_TEST_BLEU = FINAL["test"]["bleu"]
print("\n" + "=" * 60)
print("FINAL SCOREBOARD (best-valid checkpoint)")
print("=" * 60)
print("ARCH:", RESULT["architecture"])
print("TRAIN_HP:", RESULT["training_hp"])
print("seed / mode:", RESULT["seed"], "/", RESULT["mode"])
print("updates / best step:", RESULT["completed_steps"], "/", FINAL["selected_step"])
print(
    f"valid loss: {FINAL_VALID_LOSS:.4f} | BLEU: {FINAL_VALID_BLEU:.2f} | chrF: {FINAL['validation']['chrf']:.2f}"
)
print(
    f"test  loss: {FINAL_TEST_LOSS:.4f} | BLEU: {FINAL_TEST_BLEU:.2f} | chrF: {FINAL['test']['chrf']:.2f}"
)
print(f"parameters: {RESULT['parameters']:,}")
print(f"train + periodic validation: {RESULT['train_seconds'] + RESULT['eval_seconds']:.1f}s")
print("run_id:", RESULT["run_id"])
print("종료 사유:", FINAL["stop_reason"])
if not FINAL["training_completed"]:
    print("QUICK 결과: full 정상 종료한 공식 제출 점수가 아닙니다.")


In [ ]:
# SUBMISSION = export_submission(RESULT['run_id'])


### (추가) 로그 곡선 시각화
- validation BLEU·chrF와 학습 loss를 확인합니다.
- Loss는 평가 직전 update의 값이며 epoch 평균이 아닙니다.


In [ ]:
import matplotlib.pyplot as plt

history = RESULT["history"]
if history:
    steps = [row["step"] for row in history]
    for key, title, ylabel in [
        ("loss", "Train loss (logged update)", "loss"),
        ("bleu", "Validation BLEU", "BLEU"),
        ("chrf", "Validation chrF", "chrF"),
    ]:
        plt.figure()
        plt.plot(steps, [row[key] for row in history], marker="o")
        plt.title(title)
        plt.xlabel("optimizer update")
        plt.ylabel(ylabel)
        plt.grid(True, alpha=0.3)
        plt.show()


### (추가) 번역 예시 확인 (test 일부)
- 최고 validation checkpoint의 독일어 입력·영어 생성·정답을 비교합니다.


In [ ]:
print(f"Best validation checkpoint: step {FINAL['selected_step']}")
for index, example in enumerate(FINAL["test"]["examples"], start=1):
    print(f"\n[{index}]")
    print("Source:    ", example["source"])
    print("Prediction:", example["prediction"])
    print("Reference: ", example["reference"])
